# Heatwave Algorithm: Single Location Test
# ERA5 Climate Data Extraction: Regional Masking

**Final Project:** Heatwave Analysis in Rio de Janeiro State

**Course:** FA25-BL-EAS-G690-29302

**Professor:** Travis Allen O'Brien

**Step:** Heatwave Calculations

**Author:** Rafaela Quintella Veiga

**Date:** December 2025

---

## Overview
This notebook focuses on validating the `HeatwaveDetector` algorithm implemented in `core_heatwave.py`.

We use previously processed ERA5 time series (state-mean and municipal-mean 2-m air temperature) to:

- Apply the heatwave detection algorithm to a **regional** time series (state average of Rio de Janeiro).
- Apply the same algorithm to a **local** time series (Petrópolis municipality).
- Inspect the resulting **event-level table** and **summary metrics**.
- Check whether the detected frequency, duration and intensity of events are physically reasonable.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# --- Custom Library Setup ---
# Add current directory to path to find 'core_heatwave.py'
sys.path.append(os.getcwd())
import core_heatwave as hw

print(f"✅ Environment ready. Working Directory: {os.getcwd()}")

✅ Environment ready. Working Directory: c:\Users\rafaq\EASG690\Final_Project\scripts\heatwave_analysis


## 1. Configuration: Inputs & Parameters

Here we define the input files for the processed CSV data (State and Municipalities) and the global parameters used by the heatwave analysis.

- **Reference period (baseline):** 1981–2010 (standard climatological baseline)  
- **Variable:** `t2m` (2-m air temperature, °C)  
- **Time column:** `time` (daily resolution)

> Tip: In a production environment, consider using relative paths instead of absolute paths so this notebook can be executed from different machines or directories.


In [ ]:
# Configuration: File Paths
PATH_STATE = r"C:\Users\rafaq\EASG690\Final_Project\scripts\\heatwave_analysis\processed_data\results_rj_state.csv"
PATH_MUNI = r"C:\Users\rafaq\EASG690\Final_Project\scripts\heatwave_analysis\processed_data\results_rj_municipalities.csv"

# Global Analysis Parameters
REF_PERIOD = (1981, 2010)  
TARGET_VAR = 't2m'
DATE_COLUMN = 'time'       

## 2. Testing: State Scale (Regional Baseline)

In this step, we apply the `HeatwaveDetector` algorithm to the spatially aggregated daily time series of the entire **State of Rio de Janeiro**.

**Objective.**  
Validate the detection logic on a robust regional dataset before analyzing individual municipalities. This provides a *regional baseline* for interpreting heatwave frequency and intensity.

**Input:**  
- State-mean ERA5 `t2m` daily series (1940–2024), saved as `results_rj_state.csv`.

**Outputs:**  
- `events_rj` – detailed table with all detected heatwave events in the state.  
- `metrics_rj` – summary statistics grouped by percentile and duration category  
  (e.g., total number of events, average duration, mean/max intensity, annual frequency).


In [6]:
# 1. Load Data
if os.path.exists(PATH_STATE):
    df_rj = pd.read_csv(PATH_STATE)
    print(f"✅ State data loaded. Shape: {df_rj.shape}")
else:
    raise FileNotFoundError(f"File not found: {PATH_STATE}")

# 2. Initialize Detector
# Using the new 'date_col' argument for robustness
detector = hw.HeatwaveDetector(
    df_rj, 
    variable=TARGET_VAR, 
    date_col=DATE_COLUMN, 
    reference_period=REF_PERIOD
)

# 3. Run Analysis
# Thresholds: 80th, 90th, 95th percentiles | Min Duration: 3 days
print("⏳ Calculating state-level metrics...")
events_rj, metrics_rj = detector.analyze(percentiles=[80, 90, 95], min_duration=3)

# 4. Display Results
print("\n📊 State-Level Metrics Summary:")
display(metrics_rj)

print("\n📅 Recent Events (Sample):")
display(events_rj[['start_date', 'end_date', 'duration_days', 'max_temperature', 'intensity_max']].tail())

✅ State data loaded. Shape: (31014, 7)
⏳ Calculating state-level metrics...

📊 State-Level Metrics Summary:


,percentile,duration_category,total_events,avg_duration,avg_intensity,max_intensity,annual_frequency
0,80,3-4 days,389,3.331620,1.227999,6.528318,4.576471
1,80,5-7 days,161,5.515528,1.468757,6.953128,1.894118
2,80,>7 days,60,10.700000,1.666138,7.088921,0.705882
3,90,3-4 days,191,3.308901,1.058966,5.472226,2.247059
4,90,5-7 days,54,5.685185,1.256536,5.515536,0.635294
5,90,>7 days,12,9.666667,1.378616,5.954182,0.141176
6,95,3-4 days,81,3.320988,0.922616,4.703488,0.952941
7,95,5-7 days,18,5.444444,1.164408,4.751127,0.211765
8,95,>7 days,2,8.000000,1.866092,5.183220,0.023529



📅 Recent Events (Sample):


,start_date,end_date,duration_days,max_temperature,intensity_max
963,2024-05-11,2024-05-14,4,32.578983,4.164561
964,2024-06-14,2024-06-20,7,28.353049,0.672236
965,2024-08-17,2024-08-20,4,29.597249,0.836919
966,2024-09-09,2024-09-14,6,32.122688,2.501176
967,2024-09-25,2024-09-27,3,31.236692,1.601024


## 3. Testing: Municipal Scale (Local Analysis)

After validating the regional baseline, we apply the algorithm to a specific municipality to evaluate local-scale behavior and sensitivity.

**Target city:** Petrópolis  
- Located in the mountainous region of Rio de Janeiro state (Serra do Mar).  
- Cooler baseline climate compared to the coastal lowlands.

**Goal.**  
Assess how the heatwave detection algorithm behaves for a single grid-aggregated municipality and identify the most critical events.

**Outputs:**  
- `events_city` – table with all detected heatwave events for Petrópolis.  
- `metrics_city` – summary metrics by percentile and duration category.  
- A table with the **Top 5 most intense events**, sorted by maximum intensity, to highlight the most impactful episodes.



In [8]:
CITY_NAME = "Petrópolis"

if os.path.exists(PATH_MUNI):
    # 1. Load & Filter
    df_all_muni = pd.read_csv(PATH_MUNI)
    
    # Direct filter assuming the 'region' column exists (previously validated)
    df_city = df_all_muni[df_all_muni['region'] == CITY_NAME].copy()
    
    # Simple validation to ensure the city exists in the dataset
    if df_city.empty:
        raise ValueError(f"City '{CITY_NAME}' not found in the data.")
        
    print(f"✅ Data loaded for {CITY_NAME}. Records: {len(df_city)}")

    # 2. Initialize Detector
    detector_city = hw.HeatwaveDetector(
        df_city, 
        variable=TARGET_VAR, 
        date_col=DATE_COLUMN, 
        reference_period=REF_PERIOD
    )

    # 3. Run Analysis
    print(f"⏳ Calculating metrics for {CITY_NAME}...")
    events_city, metrics_city = detector_city.analyze(percentiles=[90, 95], min_duration=3)

    # 4. Display Results
    if not metrics_city.empty:
        print(f"\n📊 Metrics Summary ({CITY_NAME}):")
        display(metrics_city)
        
        print(f"\n🔥 Top 5 Most Intense Events:")
        # Sorting by max intensity to highlight critical events
        top_events = events_city.sort_values(by='intensity_max', ascending=False).head(5)
        cols_show = ['start_date', 'duration_days', 'max_temperature', 'intensity_max', 'season']
        display(top_events[cols_show])
    else:
        print("⚠️ No heatwaves detected with current criteria.")

else:
    print(f"❌ File not found: {PATH_MUNI}")

✅ Data loaded for Petrópolis. Records: 31014
⏳ Calculating metrics for Petrópolis...

📊 Metrics Summary (Petrópolis):


,percentile,duration_category,total_events,avg_duration,avg_intensity,max_intensity,annual_frequency
0,90,3-4 days,174,3.339080,1.251411,5.789480,2.047059
1,90,5-7 days,45,5.644444,1.584540,7.513192,0.529412
2,90,>7 days,9,8.444444,1.761127,6.243640,0.105882
3,95,3-4 days,64,3.328125,1.114078,4.954153,0.752941
4,95,5-7 days,14,5.357143,1.485224,6.597742,0.164706
5,95,>7 days,2,8.000000,1.984079,5.661602,0.023529



🔥 Top 5 Most Intense Events:


,start_date,duration_days,max_temperature,intensity_max,season
26,1961-09-24,6,37.050739,7.513192,SON
238,1961-09-25,5,37.050739,6.597742,SON
216,2023-11-11,8,36.839858,6.243640,SON
207,2020-09-30,4,35.439171,5.789480,SON
301,2023-11-11,8,36.839858,5.661602,SON
